# 🍇 Apollo AgriVerse - Master Dataset Pipeline (Grapes Focal Pivot)

**Pipeline Phase:** Data Integration & Master Synthesis  
**Target Crop:** Grapes (*Vitis vinifera*)  
**Output Directory:** `02_Datasets/Processed/Grapes/`  
**Notebook Location:** `02_Datasets/Master/build_grape_masters.ipynb`  

---

## 📌 Purpose & Overview
This Jupyter Notebook constructs the **two core master datasets** for the Apollo AgriVerse Digital Twin platform:
1. **`Master_Grapes_Agronomy_Reference.csv`**: Integrates benchmark soil chemistry (NPK, pH) and microclimate requirements for grape cultivation to power offline Machine Learning models.
2. **`Master_Grapes_Digital_Twin_Telemetry.csv`**: Integrates farm spatial metadata, grapevine growth-stage tracking, ambient microclimate telemetry, smart hydrogel polymer metrics, and mulching film degradation into an entity-anchored (`farm_id`) state vector.

### Step 1: Environment Setup & Directory Initialization
We import `pandas`, `numpy`, and `os` to handle file I/O, data manipulation, and directory management.

In [1]:
import pandas as pd
import numpy as np
import os

# Define relative paths based on project root layout
PROCESSED_DIR = "../Processed/Grapes"
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Target directory initialized: {os.path.abspath(PROCESSED_DIR)}")

Target directory initialized: d:\Internship\Apollo_AgriVerse\02_Datasets\Processed\Grapes


### Step 2: Build Master 1 - Grape Agronomy Reference Dataset
We load `crop_clean.csv` and `cleaned_soil_nutrients.csv`, filter strictly for `grapes`, standardize feature names, and export the output to `02_Datasets/Processed/Grapes/Master_Grapes_Agronomy_Reference.csv`.

In [2]:
# Load interim datasets
crop_clean = pd.read_csv("../Interim/Crop/crop_clean.csv")
soil_nutrients = pd.read_csv("../Interim/Soil/cleaned_soil_nutrients.csv")

# Filter strictly for Grapes
crop_grapes = crop_clean[crop_clean['label'].str.lower() == 'grapes'].copy()
soil_grapes = soil_nutrients[soil_nutrients['Name'].str.lower() == 'grapes'].copy()

# Standardize Column Names
crop_grapes.rename(columns={
    'label': 'crop_name', 'n': 'nitrogen_mgkg', 'p': 'phosphorus_mgkg', 
    'k': 'potassium_mgkg', 'ph': 'soil_ph', 'temperature': 'air_temp_c', 
    'humidity': 'humidity_pct', 'rainfall': 'rainfall_mm'
}, inplace=True)

soil_grapes.rename(columns={
    'Name': 'crop_name', 'Nitrogen': 'nitrogen_mgkg', 'Phosphorus': 'phosphorus_mgkg', 
    'Potassium': 'potassium_mgkg', 'pH': 'soil_ph', 'Temperature': 'air_temp_c', 
    'Rh': 'humidity_pct', 'Rainfall': 'rainfall_mm', 'Yield': 'yield_tons_ha'
}, inplace=True)

# Align schema and combine
selected_cols = ['crop_name', 'nitrogen_mgkg', 'phosphorus_mgkg', 'potassium_mgkg', 'soil_ph', 'air_temp_c', 'humidity_pct', 'rainfall_mm', 'yield_tons_ha']

for col in selected_cols:
    if col not in crop_grapes.columns: crop_grapes[col] = np.nan
    if col not in soil_grapes.columns: soil_grapes[col] = np.nan

master_agronomy = pd.concat([crop_grapes[selected_cols], soil_grapes[selected_cols]], ignore_index=True)
master_agronomy['yield_tons_ha'] = master_agronomy['yield_tons_ha'].fillna(master_agronomy['yield_tons_ha'].median())

# Export to Processed/Grapes folder
agronomy_path = os.path.join(PROCESSED_DIR, "Master_Grapes_Agronomy_Reference.csv")
master_agronomy.to_csv(agronomy_path, index=False)
print(f"Successfully saved Master 1 ({master_agronomy.shape[0]} rows, {master_agronomy.shape[1]} cols) to {agronomy_path}")

Successfully saved Master 1 (800 rows, 9 cols) to ../Processed/Grapes\Master_Grapes_Agronomy_Reference.csv


### Step 3: Build Master 2 - Grape Digital Twin Telemetry Dataset
We load the newly regenerated synthetic datasets, aggregate the 5-minute time-series telemetry streams (`sensor_stream`, `hydrogel`, `mulching`) into daily state averages per `farm_id`, and merge them with farm and crop lifecycle metadata.

In [3]:
# Load synthetic clean datasets (regenerated specifically for Grapes)
farm_meta = pd.read_csv("../Synthetic/cleaned/farm_metadata_clean.csv")
crop_life = pd.read_csv("../Synthetic/cleaned/crop_lifecycle_clean.csv")
sensor_stream = pd.read_csv("../Synthetic/cleaned/sensor_stream_clean.csv")
hydrogel = pd.read_csv("../Synthetic/cleaned/hydrogel_clean.csv")
mulching = pd.read_csv("../Synthetic/cleaned/mulching_clean.csv")

# 1. Aggregate time-series telemetry streams by farm_id
sensor_agg = sensor_stream.groupby('farm_id').agg({
    'temperature': 'mean', 
    'humidity': 'mean',
    'soil_moisture': 'mean', 
    'soil_temperature': 'mean'
}).reset_index().rename(columns={
    'temperature': 'air_temp_c', 
    'humidity': 'humidity_pct',
    'soil_moisture': 'soil_moisture_pct', 
    'soil_temperature': 'soil_temp_c'
})

hydrogel_agg = hydrogel.groupby('farm_id').agg({
    'water_storage': 'mean', 
    'release_rate': 'mean'
}).reset_index().rename(columns={
    'water_storage': 'hydrogel_water_storage_pct', 
    'release_rate': 'hydrogel_release_rate'
})

mulch_agg = mulching.groupby('farm_id').agg({
    'degradation_percent': 'mean', 
    'temperature_reduction': 'mean'
}).reset_index().rename(columns={
    'degradation_percent': 'mulch_degradation_pct', 
    'temperature_reduction': 'mulch_temp_reduction_c'
})

# 2. Relational merge anchored on primary key farm_id
master_twin = farm_meta.merge(crop_life.drop(columns=['crop_type']), on='farm_id', how='inner')
master_twin = master_twin.merge(sensor_agg, on='farm_id', how='left')
master_twin = master_twin.merge(hydrogel_agg, on='farm_id', how='left')
master_twin = master_twin.merge(mulch_agg, on='farm_id', how='left')

# 3. Export to Processed/Grapes folder
telemetry_path = os.path.join(PROCESSED_DIR, "Master_Grapes_Digital_Twin_Telemetry.csv")
master_twin.to_csv(telemetry_path, index=False)

print(f"Successfully saved Master 2 ({master_twin.shape[0]} rows, {master_twin.shape[1]} cols) to {telemetry_path}")

Successfully saved Master 2 (1000 rows, 24 cols) to ../Processed/Grapes\Master_Grapes_Digital_Twin_Telemetry.csv


### Step 4: Verification & Dataset Integrity Inspection
We reload both generated master files from `02_Datasets/Processed/Grapes/` to inspect shape, column integrity, and confirm zero missing values.

In [4]:
# Load generated master files for sanity check
df_m1 = pd.read_csv(agronomy_path)
df_m2 = pd.read_csv(telemetry_path)

print("=== MASTER 1: AGRONOMY REFERENCE SUMMARY ===")
print(f"File Path : {agronomy_path}")
print(f"Dimensions: {df_m1.shape[0]} rows × {df_m1.shape[1]} columns")
print(f"Null Count:\n{df_m1.isnull().sum()}\n")

print("=== MASTER 2: DIGITAL TWIN TELEMETRY SUMMARY ===")
print(f"File Path : {telemetry_path}")
print(f"Dimensions: {df_m2.shape[0]} rows × {df_m2.shape[1]} columns")
print(f"Null Count:\n{df_m2.isnull().sum()}")

=== MASTER 1: AGRONOMY REFERENCE SUMMARY ===
File Path : ../Processed/Grapes\Master_Grapes_Agronomy_Reference.csv
Dimensions: 800 rows × 9 columns
Null Count:
crop_name          0
nitrogen_mgkg      0
phosphorus_mgkg    0
potassium_mgkg     0
soil_ph            0
air_temp_c         0
humidity_pct       0
rainfall_mm        0
yield_tons_ha      0
dtype: int64

=== MASTER 2: DIGITAL TWIN TELEMETRY SUMMARY ===
File Path : ../Processed/Grapes\Master_Grapes_Digital_Twin_Telemetry.csv
Dimensions: 1000 rows × 24 columns
Null Count:
farm_id                       0
field_id                      0
latitude                      0
longitude                     0
area_acres                    0
soil_type                     0
crop_type                     0
installation_date             0
sowing_date                   0
crop_age_days                 0
growth_stage                  0
plant_height_cm               0
leaf_area_index               0
chlorophyll_index             0
canopy_cover_percent 